# 2b. Databricks: One-Minute Ossie Sync (Scheduled Job)

A trimmed-down companion to notebook 02, built to run unattended on a
**1-minute schedule** as a Databricks Job.

It does exactly one thing:

> If `ossie_from_snowflake.yaml` on S3 is **newer** than the last modification of
> the Metric View, re-import it and overwrite the Metric View. Otherwise do nothing.

What it deliberately does **not** do (all of that stays in notebook 02):

- no table creation or Iceberg registration
- no adding of measures
- no export back to `ossie_from_databricks.yaml`

Structure: configuration -> vendored shim -> freshness helpers -> import helpers
-> `run_once()`. Each section is a single cell so it reads top-to-bottom.


## Step 1 - Install the Apache Ossie Databricks converter

In [ ]:
%pip install "git+https://github.com/apache/ossie.git@01058aa416423cf43a74e7f9fb7f5f70981a418e#subdirectory=converters/databricks"

In [ ]:
dbutils.library.restartPython()

## Step 2 - Configuration

Identical names to notebook 02 so both can point at the same objects.
`GRACE_SECONDS` absorbs small clock skew between S3 and Unity Catalog metadata so
a single edit does not re-import on every run.

In [ ]:
CATALOG = "demos"
SCHEMA  = "ext_semantic_interop"

SF_NAMESPACE  = "DEMOS.EXT_SEMANTIC_INTEROP"
DBX_NAMESPACE = f"{CATALOG}.{SCHEMA}"
METRIC_VIEW   = f"{CATALOG}.{SCHEMA}.sales_metric_view"

S3_BUCKET     = "s3://snowflake-ossie-interop"      # <-- Replace with your S3 bucket
OSSIE_FROM_SF = f"{S3_BUCKET}/ossie/ossie_from_snowflake.yaml"

GRACE_SECONDS = 5      # ignore differences smaller than this
FORCE_SYNC    = False  # set True to import regardless of timestamps

print(f"Watching     : {OSSIE_FROM_SF}")
print(f"Target view  : {METRIC_VIEW}")

## Step 3 - The spec-version and dialect shim

Verbatim copy of the shim from notebook 02 (Snowflake Ossie 0.1.1 <-> Apache
converter 0.2.0.dev0). Only `snowflake_to_converter` is used here; the reverse
direction is kept so the two notebooks stay diffable.

In [ ]:
import json
import re
import yaml

CONVERTER_OSSIE_VERSION = "0.2.0.dev0"
SNOWFLAKE_OSSIE_VERSION = "0.1.1"
SNOWFLAKE_DIALECT = "SNOWFLAKE"
ANSI_DIALECT = "ANSI_SQL"
DATABRICKS_DIALECT = "DATABRICKS"


def _relabel_dialects(expression_obj, frm, to):
    if not isinstance(expression_obj, dict):
        return
    for d in expression_obj.get("dialects", []) or []:
        if d.get("dialect") == frm:
            d["dialect"] = to


def snowflake_to_converter(ossie_yaml, drop_fact_fields=True):
    root = yaml.safe_load(ossie_yaml)
    root["version"] = CONVERTER_OSSIE_VERSION
    for model in root.get("semantic_model", []) or []:
        hoisted = []
        for ds in model.get("datasets", []) or []:
            ds_name = ds.get("name", "")
            qual = re.compile(re.escape(ds_name) + r"\.", re.IGNORECASE)
            kept_ext = []
            for ext in ds.get("custom_extensions", []) or []:
                if ext.get("vendor_name") == SNOWFLAKE_DIALECT:
                    blob = json.loads(ext.get("data") or "{}")
                    for m in blob.get("metrics", []) or []:
                        expr = qual.sub("", m["expr"])
                        hoisted.append({"name": m["name"], "expression": {"dialects": [{"dialect": ANSI_DIALECT, "expression": expr}]}})
                else:
                    kept_ext.append(ext)
            if kept_ext:
                ds["custom_extensions"] = kept_ext
            else:
                ds.pop("custom_extensions", None)
            new_fields = []
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), SNOWFLAKE_DIALECT, ANSI_DIALECT)
                f.pop("custom_extensions", None)
                if drop_fact_fields and "dimension" not in f:
                    continue
                new_fields.append(f)
            if new_fields:
                ds["fields"] = new_fields
            else:
                ds.pop("fields", None)
        if hoisted:
            model["metrics"] = (model.get("metrics", []) or []) + hoisted
    # Break shared object references to prevent YAML anchor/alias syntax (&id001/*id001)
    # which Snowflake's parser cannot handle.
    root = json.loads(json.dumps(root))
    return yaml.safe_dump(root, sort_keys=False)


def converter_to_snowflake(ossie_yaml, dialect=SNOWFLAKE_DIALECT, model_name=None):
    root = yaml.safe_load(ossie_yaml)
    root["version"] = SNOWFLAKE_OSSIE_VERSION
    for model in root.get("semantic_model", []) or []:
        datasets = model.get("datasets", []) or []
        name_map = {}
        for ds in datasets:
            old_name = ds["name"]
            ds["name"] = old_name.upper()
            if old_name != ds["name"]:
                name_map[old_name] = ds["name"]
        for rel in model.get("relationships", []) or []:
            if "from" in rel:
                rel["from"] = rel["from"].upper()
            if "to" in rel:
                rel["to"] = rel["to"].upper()
        # Restore primary_key on datasets (Snowflake requires it for relationship validation).
        # First: convert unique_keys back to primary_key if present.
        for ds in datasets:
            if "unique_keys" in ds and ds["unique_keys"]:
                ds["primary_key"] = ds["unique_keys"][0]
                del ds["unique_keys"]
        fact_ds_name = datasets[0]["name"] if datasets else None
        for ds in datasets:
            is_fact = ds.get("name") == fact_ds_name
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), DATABRICKS_DIALECT, dialect)
                if not is_fact:
                    f.setdefault("dimension", {})
        fact_cols = []
        ref_re = re.compile(re.escape(fact_ds_name) + r"\.([A-Za-z_]\w*)") if fact_ds_name else None
        for m in model.get("metrics", []) or []:
            _relabel_dialects(m.get("expression"), DATABRICKS_DIALECT, dialect)
            if not fact_ds_name:
                continue
            for d in (m.get("expression") or {}).get("dialects", []) or []:
                if "expression" in d:
                    for old, new in name_map.items():
                        d["expression"] = re.sub(r"\b" + re.escape(old) + r"\.", new + ".", d["expression"])
                    d["expression"] = _qualify_columns(d["expression"], fact_ds_name)
                    for c in ref_re.findall(d["expression"]):
                        if c not in fact_cols:
                            fact_cols.append(c)
        if fact_ds_name and fact_cols:
            fact_ds = datasets[0]
            existing = {f["name"].lower() for f in fact_ds.get("fields", []) or []}
            flds = fact_ds.setdefault("fields", [])
            for c in fact_cols:
                if c.lower() not in existing:
                    flds.append({"name": c.upper(), "expression": {"dialects": [{"dialect": dialect, "expression": c}]}})
        if model_name:
            model["name"] = model_name
    # Break shared object references to prevent YAML anchor/alias syntax (&id001/*id001)
    # which Snowflake's parser cannot handle.
    root = json.loads(json.dumps(root))
    return yaml.safe_dump(root, sort_keys=False)


def _qualify_columns(expr, table):
    return re.sub(r"(?<![\w.])([A-Za-z_]\w*)(?!\s*\()(?![\w.])", lambda m: f"{table}.{m.group(1)}", expr)

## Step 4 - Freshness helpers

Two timestamps, both in epoch seconds:

- **S3 object time** from `dbutils.fs.ls`, which reports `modificationTime` in ms.
- **Metric View time** from `information_schema.tables.last_altered`, with a
  `DESCRIBE TABLE EXTENDED` fallback for workspaces where that column is not
  populated for metric views. If neither is available the view is treated as
  missing, which forces an import.

In [ ]:
from datetime import datetime, timezone


def s3_last_modified(path):
    """Epoch seconds of an S3 object, or None if it does not exist."""
    try:
        entries = dbutils.fs.ls(path)
    except Exception:
        return None
    return entries[0].modificationTime / 1000.0 if entries else None


def _last_altered_from_information_schema(catalog, schema, table):
    rows = spark.sql(f"""
        SELECT last_altered
          FROM {catalog}.information_schema.tables
         WHERE table_schema = '{schema}' AND table_name = '{table}'
    """).collect()
    return rows[0][0] if rows and rows[0][0] else None


def _last_altered_from_describe(fqname):
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {fqname}").collect()
    wanted = {"created time", "last access", "last modified"}
    for row in rows:
        if (row[0] or "").strip().lower() in wanted:
            try:
                return datetime.fromisoformat(str(row[1]).strip())
            except ValueError:
                continue
    return None


def metric_view_last_modified(fqname):
    """Epoch seconds of the Metric View's last modification, or None if absent."""
    catalog, schema, table = fqname.split(".")
    for lookup in (lambda: _last_altered_from_information_schema(catalog, schema, table),
                   lambda: _last_altered_from_describe(fqname)):
        try:
            stamp = lookup()
        except Exception:
            continue
        if stamp:
            if stamp.tzinfo is None:
                stamp = stamp.replace(tzinfo=timezone.utc)
            return stamp.timestamp()
    return None


def fmt(epoch):
    if epoch is None:
        return "n/a"
    return datetime.fromtimestamp(epoch, timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

## Step 5 - Import helpers

The import path is the same three transforms notebook 02 performs, factored into
named steps: shim the envelope, run the Apache converter, then localise the
Snowflake namespace and drop join fields older serdes reject.

In [ ]:
from ossie_databricks import convert_ossie_to_metric_view

UNSUPPORTED_JOIN_FIELDS = ("rely",)


def read_ossie(path):
    """Read Ossie YAML from S3, repairing backslash doubling from cross-platform transfer."""
    text = dbutils.fs.head(path)
    try:
        yaml.safe_load(text)
    except yaml.YAMLError:
        text = text.replace("\\\\", "\\")
        yaml.safe_load(text)
    return text


def strip_unsupported_fields(mv_yaml_text):
    mv = yaml.safe_load(mv_yaml_text)

    def clean(joins):
        for join in joins or []:
            for field in UNSUPPORTED_JOIN_FIELDS:
                join.pop(field, None)
            clean(join.get("joins"))

    clean(mv.get("joins"))
    return yaml.safe_dump(mv, sort_keys=False)


def ossie_to_metric_view_yaml(ossie_yaml):
    """Snowflake Ossie YAML -> Databricks Metric View YAML body."""
    mv_yaml = convert_ossie_to_metric_view(snowflake_to_converter(ossie_yaml))
    mv_yaml = mv_yaml.replace(SF_NAMESPACE, DBX_NAMESPACE)
    return strip_unsupported_fields(mv_yaml)


def overwrite_metric_view(fqname, mv_yaml):
    spark.sql(
        "CREATE OR REPLACE VIEW " + fqname
        + " WITH METRICS LANGUAGE YAML AS $$\n" + mv_yaml + "\n$$"
    )

## Step 6 - `run_once()`

The job entry point. Returns a small dict describing what it decided, and exits
the notebook with that dict as JSON so the outcome is visible in the Jobs run
history without opening the log.

In [ ]:
def decide(source_epoch, view_epoch):
    """Return (should_sync, reason) for a pair of timestamps."""
    if FORCE_SYNC:
        return True, "FORCE_SYNC is set"
    if source_epoch is None:
        return False, "Ossie file not found on S3"
    if view_epoch is None:
        return True, "Metric View does not exist yet"
    if source_epoch > view_epoch + GRACE_SECONDS:
        return True, "Ossie file is newer than the Metric View"
    return False, "Metric View is already up to date"


def run_once():
    source_epoch = s3_last_modified(OSSIE_FROM_SF)
    view_epoch = metric_view_last_modified(METRIC_VIEW)
    should_sync, reason = decide(source_epoch, view_epoch)

    print(f"Ossie on S3    : {fmt(source_epoch)}")
    print(f"Metric View    : {fmt(view_epoch)}")
    print(f"Decision       : {'SYNC' if should_sync else 'SKIP'} - {reason}")

    result = {
        "synced": should_sync,
        "reason": reason,
        "ossie_modified": fmt(source_epoch),
        "metric_view_modified": fmt(view_epoch),
        "metric_view": METRIC_VIEW,
    }
    if not should_sync:
        return result

    mv_yaml = ossie_to_metric_view_yaml(read_ossie(OSSIE_FROM_SF))
    overwrite_metric_view(METRIC_VIEW, mv_yaml)
    print(f"Overwrote {METRIC_VIEW}")
    result["measures"] = [m["name"] for m in yaml.safe_load(mv_yaml).get("measures", [])]
    print("Measures       :", ", ".join(result["measures"]) or "(none)")
    return result


result = run_once()

In [ ]:
import json

dbutils.notebook.exit(json.dumps(result))

## Scheduling this notebook

Workflows -> Jobs -> Create job:

- **Task type**: Notebook, pointing at this notebook (`02b_databricks_ossie_sync`)
- **Schedule**: cron `0 * * * * ?` (every minute) or a 1-minute trigger
- **Compute**: a small single-node cluster, or serverless
- **Max concurrent runs**: 1, so a slow run never overlaps the next tick

Because the notebook installs the converter with `%pip`, prefer a cluster that
stays warm between runs; otherwise most of each minute is spent on the install.
For a tighter demo loop, install `ossie-databricks` as a cluster library and
delete the `%pip` cell.

Verify the loop end-to-end by editing the Snowflake Semantic View, letting
notebook 01's export task push a new `ossie_from_snowflake.yaml`, then watching
the next run of this job report `SYNC`.
